# Étape 3 - Exploration des sources avec PySpark (UC-06)

Pour chaque source retenue à l'étape 2 (`store_sales`, `item`, `store`) :
charger, afficher le schéma et un échantillon, compter, examiner les colonnes,
identifier les valeurs manquantes, rechercher les doublons et repérer les valeurs atypiques.

Les fichiers sources sont seulement lus, jamais modifiés.

> L'inférence des types sert à une première exploration. Ce n'est pas une
> validation du schéma : les types définitifs seront imposés en Silver.

## 0. Préparation

In [15]:
import json
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Racine du projet : on remonte depuis le dossier du notebook
RACINE = os.getcwd()
# Racine du depot = dossier qui contient config/parametres.json
while not os.path.exists(os.path.join(RACINE, "config", "parametres.json")):
    if os.path.dirname(RACINE) == RACINE:
        raise RuntimeError("config/parametres.json introuvable : lancer le notebook depuis le depot")
    RACINE = os.path.dirname(RACINE)

# Sans le prefixe file://, Spark irait chercher les fichiers dans HDFS
LOCAL = "file://"
SOURCE = os.path.expanduser("~/DEV/tpcxbb_source/retail_project/data")
SCHEMA_SPEC = os.path.join(RACINE, "01_UC01_besoin_exploration", "preuves", "tpcxbb_schema.json")

# Noms de colonnes et types attendus (specification TPCx-BB, Jour 1)
with open(SCHEMA_SPEC) as f:
    SPEC = json.load(f)

spark = (
    SparkSession.builder
    .appName("RetailIntelligence-Exploration")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

In [16]:
def charger(table):
    """Charge un fichier .dat TPCx-BB : texte delimite par |, sans en-tete, encodage UTF-8."""
    noms = [c[0] for c in SPEC[table]]
    df = (
        spark.read
        .option("header", False)
        .option("sep", "|")
        .option("encoding", "UTF-8")
        .option("inferSchema", True)
        .csv(LOCAL + os.path.join(SOURCE, table, f"{table}.dat"))
    )
    # Les fichiers n'ont pas d'en-tete : on applique les noms de la specification
    return df.select(df.columns[: len(noms)]).toDF(*noms)


def types_compares(table, df):
    """Type attendu (specification) / type infere par Spark, pour chaque colonne."""
    attendus = {c[0]: c[1] for c in SPEC[table]}
    for nom, infere in df.dtypes:
        ecart = "" if attendus[nom].split("(")[0] in (infere, "bigint" if infere == "int" else "") else "  <-- ecart"
        print(f"{nom:<24} attendu {attendus[nom]:<14} infere {infere:<8}{ecart}")


def valeurs_manquantes(df):
    """Compte les null et les chaines vides de chaque colonne (une seule passe)."""
    res = df.select([
        F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]).first().asDict()
    nb = df.count()
    manquants = {c: v for c, v in res.items() if v}
    for c, v in manquants.items():
        print(f"{c:<24} {v:>8} valeur(s) manquante(s)  ({100 * v / nb:.2f} %)")
    if not manquants:
        print("Aucune valeur manquante")
    return manquants


def doublons(df, cle):
    """Lignes entierement identiques, puis doublons sur la cle candidate."""
    nb = df.count()
    exacts = nb - df.dropDuplicates().count()
    sur_cle = nb - df.dropDuplicates(cle).count()
    print("Lignes entierement dupliquees :", exacts)
    print(f"Doublons sur la cle {cle} :", sur_cle)
    return exacts, sur_cle


def valeurs_atypiques(df, colonnes):
    """Methode des quartiles (IQR) : valeur atypique si < Q1 - 1,5 x IQR ou > Q3 + 1,5 x IQR."""
    for c in colonnes:
        q1, med, q3 = df.approxQuantile(c, [0.25, 0.5, 0.75], 0.001)
        iqr = q3 - q1
        bas, haut = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        mini, maxi = df.agg(F.min(c), F.max(c)).first()
        nb = df.filter((F.col(c) < bas) | (F.col(c) > haut)).count()
        print(f"{c:<22} min {mini:>10} | Q1 {q1:>9} | mediane {med:>9} | Q3 {q3:>9} | max {maxi:>10} | atypiques {nb}")

In [17]:
# Table des dates : sert a traduire les cles *_date_sk en vraies dates
date_dim = charger("date_dim").select("d_date_sk", "d_date")

## 1. `store_sales` - ventes en magasin

### Chargement et schéma

In [18]:
ventes = charger("store_sales").cache()
ventes.printSchema()

[Stage 154:=================================================>       (7 + 1) / 8]

root
 |-- ss_sold_date_sk: integer (nullable = true)
 |-- ss_sold_time_sk: integer (nullable = true)
 |-- ss_item_sk: integer (nullable = true)
 |-- ss_customer_sk: integer (nullable = true)
 |-- ss_cdemo_sk: integer (nullable = true)
 |-- ss_hdemo_sk: integer (nullable = true)
 |-- ss_addr_sk: integer (nullable = true)
 |-- ss_store_sk: integer (nullable = true)
 |-- ss_promo_sk: integer (nullable = true)
 |-- ss_ticket_number: integer (nullable = true)
 |-- ss_quantity: integer (nullable = true)
 |-- ss_wholesale_cost: double (nullable = true)
 |-- ss_list_price: double (nullable = true)
 |-- ss_sales_price: double (nullable = true)
 |-- ss_ext_discount_amt: double (nullable = true)
 |-- ss_ext_sales_price: double (nullable = true)
 |-- ss_ext_wholesale_cost: double (nullable = true)
 |-- ss_ext_list_price: double (nullable = true)
 |-- ss_ext_tax: double (nullable = true)
 |-- ss_coupon_amt: double (nullable = true)
 |-- ss_net_paid: double (nullable = true)
 |-- ss_net_paid_inc_tax

In [5]:
types_compares("store_sales", ventes)

ss_sold_date_sk          attendu bigint         infere int     
ss_sold_time_sk          attendu bigint         infere int     
ss_item_sk               attendu bigint         infere int     
ss_customer_sk           attendu bigint         infere int     
ss_cdemo_sk              attendu bigint         infere int     
ss_hdemo_sk              attendu bigint         infere int     
ss_addr_sk               attendu bigint         infere int     
ss_store_sk              attendu bigint         infere int     
ss_promo_sk              attendu bigint         infere int     
ss_ticket_number         attendu bigint         infere int     
ss_quantity              attendu int            infere int     
ss_wholesale_cost        attendu decimal(7,2)   infere double    <-- ecart
ss_list_price            attendu decimal(7,2)   infere double    <-- ecart
ss_sales_price           attendu decimal(7,2)   infere double    <-- ecart
ss_ext_discount_amt      attendu decimal(7,2)   infere double    <-- ec

### Échantillon et nombre de lignes

In [6]:
ventes.show(10, truncate=False)
print("Nombre de lignes :", ventes.count())

+---------------+---------------+----------+--------------+-----------+-----------+----------+-----------+-----------+----------------+-----------+-----------------+-------------+--------------+-------------------+------------------+---------------------+-----------------+----------+-------------+-----------+-------------------+-------------+
|ss_sold_date_sk|ss_sold_time_sk|ss_item_sk|ss_customer_sk|ss_cdemo_sk|ss_hdemo_sk|ss_addr_sk|ss_store_sk|ss_promo_sk|ss_ticket_number|ss_quantity|ss_wholesale_cost|ss_list_price|ss_sales_price|ss_ext_discount_amt|ss_ext_sales_price|ss_ext_wholesale_cost|ss_ext_list_price|ss_ext_tax|ss_coupon_amt|ss_net_paid|ss_net_paid_inc_tax|ss_net_profit|
+---------------+---------------+----------+--------------+-----------+-----------+----------+-----------+-----------+----------------+-----------+-----------------+-------------+--------------+-------------------+------------------+---------------------+-----------------+----------+-------------+-----------+

[Stage 5:==============>                                            (2 + 6) / 8]

Nombre de lignes : 667579


### Examen des colonnes

In [7]:
# Nombre de valeurs distinctes par colonne
ventes.agg(*[F.countDistinct(c).alias(c) for c in ventes.columns]).show(vertical=True)

[Stage 11:=======>                                                  (1 + 7) / 8]

-RECORD 0-----------------------
 ss_sold_date_sk       | 1808   
 ss_sold_time_sk       | 55645  
 ss_item_sk            | 17820  
 ss_customer_sk        | 37336  
 ss_cdemo_sk           | 36953  
 ss_hdemo_sk           | 7161   
 ss_addr_sk            | 26286  
 ss_store_sk           | 12     
 ss_promo_sk           | 300    
 ss_ticket_number      | 89100  
 ss_quantity           | 100    
 ss_wholesale_cost     | 9901   
 ss_list_price         | 19524  
 ss_sales_price        | 17199  
 ss_ext_discount_amt   | 238093 
 ss_ext_sales_price    | 237770 
 ss_ext_wholesale_cost | 264513 
 ss_ext_list_price     | 328908 
 ss_ext_tax            | 53387  
 ss_coupon_amt         | 87123  
 ss_net_paid           | 255026 
 ss_net_paid_inc_tax   | 315642 
 ss_net_profit         | 288874 



In [8]:
# Statistiques des colonnes numeriques de mesure
MESURES_VENTES = ["ss_quantity", "ss_wholesale_cost", "ss_list_price", "ss_sales_price",
                  "ss_ext_discount_amt", "ss_ext_sales_price", "ss_coupon_amt", "ss_net_paid", "ss_net_profit"]
ventes.select(MESURES_VENTES).summary("count", "mean", "min", "25%", "50%", "75%", "max").show(truncate=False)

[Stage 17:>                                                         (0 + 1) / 1]

+-------+------------------+------------------+-----------------+-----------------+-------------------+------------------+------------------+------------------+------------------+
|summary|ss_quantity       |ss_wholesale_cost |ss_list_price    |ss_sales_price   |ss_ext_discount_amt|ss_ext_sales_price|ss_coupon_amt     |ss_net_paid       |ss_net_profit     |
+-------+------------------+------------------+-----------------+-----------------+-------------------+------------------+------------------+------------------+------------------+
|count  |667579            |667579            |667579           |667579           |667579             |667579            |667579            |667579            |667579            |
|mean   |50.564014146640325|50.488922689299756|75.72587788111989|37.86369706057271|1912.4542140031415 |1913.2177683989469|189.78191081504994|1723.4358575839005|-827.6973583950385|
|min    |1                 |1.0               |1.0              |0.0              |0.0              

In [9]:
# Periode couverte : on traduit ss_sold_date_sk en date grace a date_dim
(ventes.join(date_dim, ventes.ss_sold_date_sk == date_dim.d_date_sk, "left")
       .agg(F.min("d_date").alias("premiere_vente"), F.max("d_date").alias("derniere_vente"),
            F.sum(F.when(F.col("d_date").isNull(), 1).otherwise(0)).alias("cles_date_inconnues"))
       .show())

+--------------+--------------+-------------------+
|premiere_vente|derniere_vente|cles_date_inconnues|
+--------------+--------------+-------------------+
|    2001-01-01|    2005-12-13|                  0|
+--------------+--------------+-------------------+



### Valeurs manquantes

In [10]:
manquants_ventes = valeurs_manquantes(ventes)

[Stage 22:=======>                                                  (1 + 7) / 8]

ss_cdemo_sk                   238 valeur(s) manquante(s)  (0.04 %)
ss_hdemo_sk                   295 valeur(s) manquante(s)  (0.04 %)
ss_addr_sk                    448 valeur(s) manquante(s)  (0.07 %)
ss_promo_sk                   158 valeur(s) manquante(s)  (0.02 %)


### Doublons

In [11]:
doublons_ventes = doublons(ventes, ["ss_ticket_number", "ss_item_sk"])

Lignes entierement dupliquees : 0
Doublons sur la cle ['ss_ticket_number', 'ss_item_sk'] : 4136


In [12]:
# Exemple de doublon sur (ticket, produit) : les lignes different-elles ?
exemple = (ventes.groupBy("ss_ticket_number", "ss_item_sk").count()
                 .filter("count > 1").orderBy("ss_ticket_number").first())
(ventes.filter((F.col("ss_ticket_number") == exemple.ss_ticket_number) & (F.col("ss_item_sk") == exemple.ss_item_sk))
       .select("ss_ticket_number", "ss_item_sk", "ss_sold_date_sk", "ss_quantity", "ss_sales_price", "ss_net_paid")
       .show())

+----------------+----------+---------------+-----------+--------------+-----------+
|ss_ticket_number|ss_item_sk|ss_sold_date_sk|ss_quantity|ss_sales_price|ss_net_paid|
+----------------+----------+---------------+-----------+--------------+-----------+
|              44|     10003|          36890|         31|         53.16|    1647.96|
|              44|     10003|          36890|         46|           1.6|       5.89|
+----------------+----------+---------------+-----------+--------------+-----------+



### Valeurs atypiques

In [13]:
valeurs_atypiques(ventes, MESURES_VENTES)

ss_quantity            min          1 | Q1      26.0 | mediane      51.0 | Q3      76.0 | max        100 | atypiques 0
ss_wholesale_cost      min        1.0 | Q1      25.7 | mediane     50.49 | Q3      75.2 | max      100.0 | atypiques 0
ss_list_price          min        1.0 | Q1     37.03 | mediane     72.78 | Q3    109.07 | max      200.0 | atypiques 0
ss_sales_price         min        0.0 | Q1     10.25 | mediane     27.66 | Q3     56.69 | max      199.3 | atypiques 15467
ss_ext_discount_amt    min        0.0 | Q1    312.96 | mediane   1045.74 | Q3   2663.76 | max    18984.0 | atypiques 42142
ss_ext_sales_price     min        0.0 | Q1    313.02 | mediane   1046.76 | Q3   2669.28 | max    18745.0 | atypiques 41912
ss_coupon_amt          min        0.0 | Q1       0.0 | mediane       0.0 | Q3       0.0 | max   14371.66 | atypiques 137194
ss_net_paid            min        0.0 | Q1    245.14 | mediane    877.66 | Q3   2361.64 | max    18745.0 | atypiques 46510
ss_net_profit          min 

In [14]:
# Incoherences au regard du sens des colonnes
controles_ventes = {
    "quantite <= 0": ventes.filter(F.col("ss_quantity") <= 0).count(),
    "prix de vente < 0": ventes.filter(F.col("ss_sales_price") < 0).count(),
    "prix de vente = 0": ventes.filter(F.col("ss_sales_price") == 0).count(),
    "prix de vente > prix catalogue": ventes.filter(F.col("ss_sales_price") > F.col("ss_list_price")).count(),
    "montant != quantite x prix": ventes.filter(
        F.abs(F.col("ss_ext_sales_price") - F.col("ss_quantity") * F.col("ss_sales_price")) > 0.01).count(),
    "remise > montant catalogue": ventes.filter(F.col("ss_ext_discount_amt") > F.col("ss_ext_list_price")).count(),
    "coupon > montant vendu": ventes.filter(F.col("ss_coupon_amt") > F.col("ss_ext_sales_price")).count(),
    "marge negative": ventes.filter(F.col("ss_net_profit") < 0).count(),
    "vente sous le cout d'achat": ventes.filter(F.col("ss_sales_price") < F.col("ss_wholesale_cost")).count(),
}
for k, v in controles_ventes.items():
    print(f"{k:<32} {v:>8} ligne(s)")

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/home/lolorigolo/DEV/J1_Retail_Intelligence/tp3_lolo/.venv/lib/python3.14/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/lolorigolo/DEV/J1_Retail_Intelligence/tp3_lolo/.venv/lib/python3.14/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/lib/python3.14/socket.py", line 712, in readinto
    def readinto(self, b):
    
KeyboardInterrupt


KeyboardInterrupt: 

## 2. `item` - produits

### Chargement et schéma

In [ ]:
produits = charger("item").cache()
produits.printSchema()

In [ ]:
types_compares("item", produits)

### Échantillon et nombre de lignes

In [ ]:
produits.drop("i_item_desc").show(10, truncate=False)
print("Nombre de lignes :", produits.count())

### Examen des colonnes

In [ ]:
produits.agg(*[F.countDistinct(c).alias(c) for c in produits.columns]).show(vertical=True)

In [ ]:
# Repartition par categorie
produits.groupBy("i_category_id", "i_category").count().orderBy("i_category_id").show(truncate=False)

In [ ]:
MESURES_PRODUITS = ["i_current_price", "i_wholesale_cost"]
produits.select(MESURES_PRODUITS).summary("count", "mean", "min", "25%", "50%", "75%", "max").show()
produits.agg(F.min("i_rec_start_date"), F.max("i_rec_start_date")).show()

### Valeurs manquantes

In [ ]:
manquants_produits = valeurs_manquantes(produits)

### Doublons

In [ ]:
doublons_produits = doublons(produits, ["i_item_sk"])
print("Doublons sur i_item_id :", produits.count() - produits.dropDuplicates(["i_item_id"]).count())

### Valeurs atypiques

In [ ]:
valeurs_atypiques(produits, MESURES_PRODUITS)

In [ ]:
controles_produits = {
    "prix courant <= 0": produits.filter(F.col("i_current_price") <= 0).count(),
    "cout d'achat > prix courant": produits.filter(F.col("i_wholesale_cost") > F.col("i_current_price")).count(),
    "categorie_id sans libelle unique": produits.groupBy("i_category_id")
        .agg(F.countDistinct("i_category").alias("n")).filter("n > 1").count(),
}
for k, v in controles_produits.items():
    print(f"{k:<34} {v:>6}")

## 3. `store` - magasins

### Chargement et schéma

In [ ]:
magasins = charger("store").cache()
magasins.printSchema()

In [ ]:
types_compares("store", magasins)

### Échantillon et nombre de lignes

In [ ]:
magasins.select("s_store_sk", "s_store_id", "s_store_name", "s_rec_start_date", "s_closed_date_sk",
                "s_number_employees", "s_floor_space", "s_hours", "s_city", "s_state", "s_zip",
                "s_gmt_offset", "s_tax_precentage").show(12, truncate=False)
print("Nombre de lignes :", magasins.count())

### Examen des colonnes

In [ ]:
magasins.agg(*[F.countDistinct(c).alias(c) for c in magasins.columns]).show(vertical=True)

In [ ]:
MESURES_MAGASINS = ["s_number_employees", "s_floor_space", "s_tax_precentage"]
magasins.select(MESURES_MAGASINS).summary("count", "mean", "min", "max").show()

### Valeurs manquantes

In [ ]:
manquants_magasins = valeurs_manquantes(magasins)

### Doublons

In [ ]:
doublons_magasins = doublons(magasins, ["s_store_sk"])
print("Doublons sur s_store_id :", magasins.count() - magasins.dropDuplicates(["s_store_id"]).count())

### Valeurs atypiques

In [ ]:
valeurs_atypiques(magasins, MESURES_MAGASINS)

In [ ]:
# Date de fermeture : on traduit s_closed_date_sk en date
(magasins.join(date_dim, magasins.s_closed_date_sk == date_dim.d_date_sk, "left")
         .select("s_store_sk", "s_store_name", "s_rec_start_date", "s_closed_date_sk",
                 F.col("d_date").alias("date_fermeture"))
         .orderBy("date_fermeture").show(truncate=False))

In [ ]:
# Horaires d'ouverture
magasins.groupBy("s_hours").count().show()

In [ ]:
# Codes postaux : lus comme des nombres, ils perdent leurs zeros initiaux
magasins.select("s_zip", F.length(F.col("s_zip").cast("string")).alias("nb_chiffres")).groupBy("nb_chiffres").count().show()

## 4. Synthèse des contrôles

In [ ]:
synthese = {
    "store_sales": {"lignes": ventes.count(), "colonnes": len(ventes.columns), "manquants": manquants_ventes,
                    "doublons_exacts": doublons_ventes[0], "doublons_cle": doublons_ventes[1],
                    "controles": controles_ventes},
    "item": {"lignes": produits.count(), "colonnes": len(produits.columns), "manquants": manquants_produits,
             "doublons_exacts": doublons_produits[0], "doublons_cle": doublons_produits[1],
             "controles": controles_produits},
    "store": {"lignes": magasins.count(), "colonnes": len(magasins.columns), "manquants": manquants_magasins,
              "doublons_exacts": doublons_magasins[0], "doublons_cle": doublons_magasins[1]},
}
print(json.dumps(synthese, indent=1, ensure_ascii=False))

## 5. Réponses aux questions techniques

**1. Quelles colonnes sont numériques ?**

- **`store_sales`** : `ss_quantity` (entier), et les 12 montants `ss_wholesale_cost`, `ss_list_price`, `ss_sales_price`, `ss_ext_*`, `ss_coupon_amt`, `ss_net_paid`, `ss_net_paid_inc_tax`, `ss_net_profit`. Spark les lit en `double`, alors que la spécification attend `decimal(7,2)`.
- **`item`** : `i_current_price` et `i_wholesale_cost`.
- **`store`** : `s_number_employees`, `s_floor_space`, `s_tax_precentage` et `s_gmt_offset`.
- **Faux numériques** : `s_zip` et `s_street_number` sont lus comme des nombres, mais ce sont des codes. Ils doivent rester du texte. Le code postal perdrait ses zéros initiaux ; ici, les 12 codes ont bien 5 chiffres.

**2. Quelles colonnes représentent des dates ?**

- **Vraies dates** : `i_rec_start_date` et `s_rec_start_date`, que Spark reconnaît comme `date`. Les colonnes `*_rec_end_date` sont toujours vides.
- **Dates codées en entiers** : `ss_sold_date_sk`, `s_closed_date_sk` et `ss_sold_time_sk` (l'heure) sont des clés vers `date_dim` et `time_dim`. Il faut une jointure pour obtenir la vraie date : les ventes vont du 01/01/2001 au 13/12/2005, et toutes les clés de date sont reconnues.

**3. Quelles colonnes sont des identifiants ?**

- Les colonnes en `*_sk` sont des clés techniques : `ss_item_sk`, `ss_store_sk`, `ss_customer_sk`, `i_item_sk`, `s_store_sk`, etc.
- Les colonnes en `*_id` sont des codes métier : `i_item_id`, `s_store_id`, `i_category_id`, etc.
- `ss_ticket_number` est le numéro de ticket de caisse.
- `store_sales` n'a pas d'identifiant de ligne qui lui soit propre.

**4. Quels champs présentent des valeurs manquantes ?**

- **`store_sales`** : `ss_cdemo_sk` (238), `ss_hdemo_sk` (295), `ss_addr_sk` (448) et `ss_promo_sk` (158). Cela fait moins de 0,1 % des lignes. Pour `ss_promo_sk`, un vide peut signifier « vente sans promotion ».
- **`item`** : `i_item_desc` (4), `i_brand_id` (8) et `i_brand` (6).
- **Colonnes entièrement vides** : `i_rec_end_date` et `s_rec_end_date`. C'est normal, car il n'y a pas d'historique.
- Aucune colonne utilisée pour le CA ou la marge n'est vide.

**5. Existe-t-il des doublons ?**

- **Copies exactes** : aucune ligne n'est entièrement dupliquée, dans aucune des 3 sources.
- **`store_sales`** : 4 136 lignes répètent un couple (ticket, produit) déjà présent, mais avec une quantité et un prix différents. Exemple : le ticket 44 contient deux fois le produit 10003, une fois 31 unités à 53,16 et une fois 46 unités à 1,60. Ce sont plutôt deux lignes de vente distinctes que des doublons techniques.
- **`item` et `store`** : leurs clés (`i_item_sk`, `i_item_id`, `s_store_sk`, `s_store_id`) sont toutes uniques.

**6. Certaines valeurs sont-elles incohérentes au regard de leur signification ?**

- **Calculs** : ils sont cohérents. Il n'y a aucune quantité nulle ou négative, aucun prix négatif, aucun prix de vente supérieur au prix catalogue, et le montant vaut toujours quantité × prix.
- **Ventes à prix nul** : 3 324 lignes ont un `ss_sales_price` à 0.
- **Ventes à perte** : 74 % des lignes ont une marge négative (495 971), et 69 % sont vendues sous le coût d'achat (462 852). Dans `item`, 7 768 produits ont un coût d'achat supérieur au prix courant. Ces chiffres sont peu réalistes pour un commerce et viennent de la génération synthétique TPCx-BB : il faut le dire avant de présenter une marge.
- **Valeurs atypiques (IQR)** : les montants dépassent souvent le 3e quartile + 1,5 × IQR, par exemple 41 912 lignes pour `ss_ext_sales_price`. Ce sont de gros paniers, pas des erreurs, car les maximums restent cohérents avec quantité × prix. Pour `ss_coupon_amt`, le Q3 vaut 0 : toute ligne avec un coupon apparaît donc comme atypique.
- **Dates de fermeture** : `s_closed_date_sk` est incohérent. Les 12 magasins ouvrent en 2000 et vendent jusqu'en 2005, mais leurs dates de fermeture vont de 1917 à 2141. Cette colonne est inutilisable.
- **Horaires** : un magasin (Buffalo) a des horaires `8AM-8AM`.

In [ ]:
spark.stop()